# SOFR, FX & Cross-Currency Curves · Episode 9 — Pricing with the curves

Companion notebook for the video: long-dated AUD/USD forwards from the curve, what ignoring the basis would cost, and the value of an off-market basis swap, two ways. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Data.** All quotes are illustrative, made up for teaching: AONIA and BBSW swaps (`quotes_illustrative.csv`, the AUD series), SOFR swaps and futures (`quotes_sofr_illustrative.csv`, Episode 3), and AUD/USD spot, FX swap points and BBSW/SOFR basis swaps (`quotes_fx_illustrative.csv`, Episodes 5-6). They are shaped like a plausible market, not market data. Educational material only, not investment advice. Rates with Bip is not affiliated with the New York Fed.

1. Setup · 2. Long-dated FX forwards · 3. An off-market basis swap · 4. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_sofr_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_sofr_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_sofr_illustrative.csv').write_text("""instrument,label,value,unit
ois,1W,3.88,pct
ois,1M,3.87,pct
ois,2M,3.86,pct
ois,3M,3.85,pct
future,SR3 Dec 2026,96.20,price
future,SR3 Mar 2027,96.30,price
future,SR3 Jun 2027,96.39,price
future,SR3 Sep 2027,96.44,price
ois,2Y,3.66,pct
ois,3Y,3.63,pct
ois,5Y,3.63,pct
ois,7Y,3.70,pct
ois,10Y,3.81,pct
ois,15Y,3.94,pct
ois,20Y,3.98,pct
ois,30Y,3.96,pct
""")
print('wrote quotes_sofr_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_fx_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_fx_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_fx_illustrative.csv').write_text("""instrument,label,value,unit
fx,spot,0.71,rate
fx,ON,-0.098,pips
fx,TN,-0.033,pips
fx,1W,-0.23,pips
fx,1M,-1.1,pips
fx,2M,-2.66,pips
fx,3M,-4.29,pips
fx,6M,-10.68,pips
fx,9M,-19.65,pips
fx,1Y,-30.32,pips
xccy,2Y,14.75,bp
xccy,3Y,14.75,bp
xccy,4Y,15.25,bp
xccy,5Y,15.5,bp
xccy,7Y,15.5,bp
xccy,10Y,15.5,bp
""")
print('wrote quotes_fx_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-25"
aud_quotes_file = "quotes_illustrative.csv"
sofr_quotes_file = "quotes_sofr_illustrative.csv"
fx_quotes_file = "quotes_fx_illustrative.csv"
output_json = "build/outputs.json"
traded_spread_bp = 12.0      # illustrative: the off-market swap
notional_aud = 100_000_000

## 1. Setup: every curve of the series

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
cal, dc = ql.Australia(), ql.Actual365Fixed()    # used by the AUD curve code
iso = lambda d: d.ISO()
out = {"meta": {"series": "SOFR, FX & Cross-Currency Curves", "episode": 9, "valuation_date": valuation_date,
                "quantlib_version": ql.__version__, "quotes_label": "illustrative (made up for teaching)",
                "generated_at": dt.datetime.now().isoformat(timespec="seconds")}}

AONIA and BBSW curves (the AUD series):

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

The SOFR curve (Episode 3):

In [ ]:
# SOFR curve used from Series 2 Episode 3 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today (the evaluation date) from the setup cell.
#   SOFR OIS: CFTC Swaps Made Available to Trade chart (USD SOFR OIS): spot T+2, annual fixed and floating
#     legs, ACT/360, payment lag 2 days, New York business days; SOFR fixes on the government securities calendar.
#   3M SOFR futures: CME Submission 18-069: SOFR compounded over the IMM-to-IMM Reference Quarter,
#     price = 100 - rate; named by the month the Reference Quarter starts. Convexity set to zero (a simplification).
USD_CAL = ql.UnitedStates(ql.UnitedStates.Settlement)  # stand-in for New York business days
USD_DC = ql.Actual360()
MONTHS = {m: i for i, m in enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                      "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

def sofr_ois_helper(tenor, quote):
    return ql.OISRateHelper(2, ql.Period(tenor), quote, ql.Sofr(), paymentLag=2,
                            paymentFrequency=ql.Annual, paymentCalendar=USD_CAL,
                            convention=ql.ModifiedFollowing, endOfMonth=True)

def build_sofr_curve(quotes):
    """quotes: rows (instrument, label, value); OIS values in %, futures as prices. Returns curve, quotes, helpers."""
    q, helpers = {}, {}
    for r in quotes.itertuples():
        if r.instrument == "future":  # label like "SR3 Dec 2026"
            q[r.label] = ql.SimpleQuote(r.value)
            _, mon, year = r.label.split()
            helpers[r.label] = ql.SofrFutureRateHelper(ql.QuoteHandle(q[r.label]), MONTHS[mon], int(year),
                                                       ql.Quarterly)
        else:
            q[r.label] = ql.SimpleQuote(r.value / 100)
            helpers[r.label] = sofr_ois_helper(r.label, ql.QuoteHandle(q[r.label]))
    curve = ql.PiecewiseLogLinearDiscount(today, list(helpers.values()), USD_DC)
    curve.enableExtrapolation()
    curve.nodes()  # bootstrap now: the helpers only see the curve once it has been built
    return curve, q, helpers

The AUD curve under USD collateral (built in Episode 8) and the basis swap valuation:

In [ ]:
# AUD discount curve under USD collateral, used from Series 2 Episode 6 on. Copied verbatim into each notebook by
# make_notebook.py, so every notebook runs on its own. Needs: ql, pd, today, cal, dc and the AUD curve code
# (aud_curves.py) and SOFR curve code (sofr_curve.py) from earlier cells.
#   Short end: AUD/USD FX swaps, overnight to one year (forward = spot + points).
#   Long end: BBSW/SOFR mark-to-market cross-currency basis swaps, two to ten years (AFMA term sheet, 2024):
#     spot T+2 on Sydney and New York days, quarterly; receive 3M BBSW + spread on a constant AUD notional,
#     pay SOFR compounded with no spread on a USD notional reset each quarter to the forward FX rate.
#   Collateral: US dollars, discounted at SOFR (AFMA). The curve solved for is the AUD one.
#   Modelling choice: QuantLib's helper pays coupons and notionals on the same dates. A payment lag of zero
#   puts the notional exchanges on the roll dates, as in the term sheet; the coupons then pay two days early,
#   which moves the fair spread by less than a tenth of a basis point here.
JOINT = ql.JointCalendar(ql.Australia(ql.Australia.Settlement), USD_CAL)
FX_TENORS = ["1W", "1M", "2M", "3M", "6M", "9M", "1Y"]

class Market:
    """All curves of the series: AONIA and BBSW (AUD series), SOFR (Episode 3), AUD under USD collateral."""

    def __init__(self, aud_quotes, sofr_quotes, fx_quotes):
        _, handles, self.aud_q, _ = build_curves(aud_quotes)
        self.aonia, self.bbsw3m = handles["AONIA"], handles["BBSW3M"]
        self.sofr_curve, self.sofr_q, _ = build_sofr_curve(sofr_quotes)
        self.sofr = ql.YieldTermStructureHandle(self.sofr_curve)
        self.tom, self.spot = JOINT.advance(today, 1, ql.Days), JOINT.advance(today, 2, ql.Days)
        fx = fx_quotes[fx_quotes.instrument == "fx"].set_index("label").value
        xc = fx_quotes[fx_quotes.instrument == "xccy"].set_index("label").value
        self.S = ql.SimpleQuote(float(fx["spot"]))
        self.points = {t: ql.SimpleQuote(float(fx[t]) / 1e4) for t in ["ON", "TN"] + FX_TENORS}
        self.basis = {t: ql.SimpleQuote(float(v) / 1e4) for t, v in xc.items()}
        self.near_on, self.near_tn = ql.SimpleQuote(0.0), ql.SimpleQuote(0.0)
        self._set_near()
        h = ql.QuoteHandle
        self.helpers = {
            "ON": ql.FxSwapRateHelper(h(self.points["ON"]), h(self.near_on), ql.Period(1, ql.Days), 0, JOINT,
                                      ql.Following, False, False, self.sofr),
            "TN": ql.FxSwapRateHelper(h(self.points["TN"]), h(self.near_tn), ql.Period(1, ql.Days), 1, JOINT,
                                      ql.Following, False, False, self.sofr)}
        for t in FX_TENORS:
            self.helpers[t] = ql.FxSwapRateHelper(h(self.points[t]), h(self.S), ql.Period(t), 2, JOINT,
                                                  ql.ModifiedFollowing, True, False, self.sofr)
        for t, q in self.basis.items():
            self.helpers["basis " + t] = ql.MtMCrossCurrencyBasisSwapRateHelper(
                h(q), ql.Period(t), 2, JOINT, ql.ModifiedFollowing, True,
                bbsw(3, self.bbsw3m), ql.Sofr(self.sofr), self.sofr,
                False,   # isFxBaseCurrencyCollateralCurrency: AUD is the base currency; collateral is USD
                True,    # isBasisOnFxBaseCurrencyLeg: the spread is on the AUD (BBSW) leg
                False,   # isFxBaseCurrencyLegResettable: the USD notional resets, the AUD one is constant
                ql.Quarterly, 0, ql.Quarterly)
        self.curve = ql.PiecewiseLogLinearDiscount(today, list(self.helpers.values()), dc)
        self.curve.enableExtrapolation()
        self.curve.nodes()
        self.aud_usd = ql.YieldTermStructureHandle(self.curve)

    def _set_near(self):  # today's and tomorrow's rates, built back from spot with the ON and TN points
        self.near_tn.setValue(self.S.value() - self.points["TN"].value())
        self.near_on.setValue(self.S.value() - self.points["TN"].value() - self.points["ON"].value())

    def set_spot(self, value):
        self.S.setValue(value)
        self._set_near()

    def fwd_fx(self, d):
        """AUD/USD outright for value date d: spot x P_aud(spot, d) / P_usd(spot, d)."""
        a, u = self.aud_usd, self.sofr
        return self.S.value() * (a.discount(d) / a.discount(self.spot)) / (u.discount(d) / u.discount(self.spot))

def mtm_swap(m, tenor, spread, aud_notional, coupon_lag=0):
    """Value a BBSW/SOFR mark-to-market basis swap to the party receiving the AUD leg (BBSW + spread).

    Dates as the curve's helpers build them: spot + tenor, rolled backwards quarterly, end-of-month rule.
    Each quarter the USD notional is the AUD notional times the forward FX rate to the period start; notionals
    move on roll dates; coupons are paid coupon_lag business days after each period end (term sheet: 2; the
    curve build: 0). Values are in AUD at today's FX rate. Returns the value, the value of one basis point of
    spread, and one row per period.
    """
    sch = list(ql.MakeSchedule(m.spot, m.spot + ql.Period(tenor), ql.Period(ql.Quarterly), calendar=JOINT,
                               convention=ql.ModifiedFollowing, endOfMonth=True, backwards=True))
    pa, pu = m.aud_usd.discount, m.sofr.discount
    fx_today = m.fwd_fx(today)
    index = bbsw(3, m.bbsw3m)
    pv_aud = -aud_notional * pa(sch[0]) + aud_notional * pa(sch[-1])
    pv_usd, bps, rows = 0.0, 0.0, []
    for i, (a, b) in enumerate(zip(sch[:-1], sch[1:])):
        pay = JOINT.advance(b, coupon_lag, ql.Days) if coupon_lag else b
        tau_a, tau_u = dc.yearFraction(a, b), USD_DC.yearFraction(a, b)
        bbsw_rate = (m.bbsw3m.discount(a) / m.bbsw3m.discount(b) - 1) / tau_a
        sofr_rate = (pu(a) / pu(b) - 1) / tau_u
        fx = m.fwd_fx(a)
        usd_n = aud_notional * fx
        aud_cpn = aud_notional * (bbsw_rate + spread) * tau_a
        usd_cpn = usd_n * sofr_rate * tau_u
        pv_aud += aud_cpn * pa(pay)
        bps += aud_notional * tau_a * pa(pay) * 1e-4
        pv_usd += usd_n * (pu(a) - pu(b)) - usd_cpn * pu(pay)   # receive the USD notional, repay it with SOFR
        rows.append({"period": i + 1, "start": a.ISO(), "end": b.ISO(), "pay": pay.ISO(),
                     "bbsw_pct": bbsw_rate * 100, "aud_rate_pct": (bbsw_rate + spread) * 100, "aud_coupon": aud_cpn,
                     "fx": fx, "usd_notional": usd_n, "sofr_pct": sofr_rate * 100, "usd_coupon": usd_cpn})
    value = pv_aud + pv_usd / fx_today
    return {"value": value, "bps": bps, "pv_aud_leg": pv_aud, "pv_usd_leg_aud": pv_usd / fx_today, "rows": rows}

def par_spread(m, tenor, coupon_lag=0):
    """Spread on the AUD leg that makes a new mark-to-market basis swap worth zero."""
    r = mtm_swap(m, tenor, 0.0, 1.0, coupon_lag)
    return -r["value"] / (r["bps"] * 1e4)

In [ ]:
m = Market(pd.read_csv(aud_quotes_file), pd.read_csv(sofr_quotes_file), pd.read_csv(fx_quotes_file))
print("today", today, "| spot", m.spot, "| AUD/USD", m.S.value(),
      "| largest repricing error", max(abs(h.impliedQuote() - h.quote().value()) for h in m.helpers.values()))

## 2. Long-dated FX forwards

FX swap quotes stop at a year; the curve goes to ten. The outright forward for any date is spot times the ratio of discount factors, $F(T) = S\,P_\text{AUD}(s,T)/P_\text{SOFR}(s,T)$, with the AUD curve under USD collateral. For comparison: the same formula with AONIA, which is what you'd get if you ignored the basis.

In [ ]:
fw = []
for y in (1, 2, 5, 10):
    T = JOINT.advance(m.spot, ql.Period(y, ql.Years), ql.ModifiedFollowing, True)
    F = m.fwd_fx(T)
    F_aonia = m.S.value() * (m.aonia.discount(T) / m.aonia.discount(m.spot)) / (m.sofr.discount(T) / m.sofr.discount(m.spot))
    fw.append({"years": y, "date": iso(T), "outright": F, "points": (F - m.S.value()) * 1e4,
               "outright_aonia": F_aonia, "diff_pips": (F - F_aonia) * 1e4})
fwd = pd.DataFrame(fw)
f5 = fwd[fwd.years == 5].iloc[0]
T5 = ql.DateParser.parseISO(f5.date)
fx_today = m.fwd_fx(today)
# A five-year forward on AUD 100 million priced with AONIA instead of the right curve: the price error, in AUD today
err_usd = notional_aud * (f5.outright_aonia - f5.outright) * m.sofr.discount(T5)
out["forwards"] = {"rows": fw, "spot": m.S.value(),
                   "f5_outright": float(f5.outright), "f5_points": float(f5.points),
                   "f5_aonia": float(f5.outright_aonia), "f5_diff_pips": float(f5.diff_pips),
                   "f10_diff_pips": float(fwd.diff_pips.iloc[-1]), "f1_diff_pips": float(fwd.diff_pips.iloc[0]),
                   "mispricing_aud": err_usd / fx_today, "notional": notional_aud}
fwd

## 3. An off-market basis swap

An Australian bank hedging its US dollar debt pays the AUD leg of a five-year BBSW/SOFR basis swap at BBSW + 12 basis points (illustrative), and receives SOFR, on AUD 100 million. Today's five-year basis is 15.5 basis points. What is the swap worth to the bank?

Two ways: value every cash flow of both legs (the Episode 6 loop), or take the difference between today's basis and the traded one, times the value of one basis point of spread. The loop uses the curve's conventions, so a swap at today's basis is worth exactly zero.

In [ ]:
market_bp = m.basis["5Y"].value() * 1e4
rec = mtm_swap(m, "5Y", traded_spread_bp / 1e4, notional_aud, coupon_lag=0)   # value to the AUD-leg receiver
value_bank = -rec["value"]                                                    # the bank pays the AUD leg
bp_value = rec["bps"]
shortcut = (market_bp - traded_spread_bp) * bp_value
par_check = mtm_swap(m, "5Y", market_bp / 1e4, notional_aud, coupon_lag=0)["value"]
out["swap"] = {"tenor": "5Y", "notional": notional_aud, "traded_bp": traded_spread_bp, "market_bp": market_bp,
               "gap_bp": market_bp - traded_spread_bp,
               "value_bank": value_bank, "bp_value": bp_value, "shortcut": shortcut,
               "abs_diff": abs(value_bank - shortcut), "value_at_market": par_check,
               "pv_aud_leg": -rec["pv_aud_leg"], "pv_usd_leg": -rec["pv_usd_leg_aud"],
               "n_periods": len(rec["rows"])}
pd.Series({k: v for k, v in out["swap"].items()})

## 4. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")